# Diabetic Retinopathy Screening Agent
Vicheda Narith, Maanvi Sarwadi

## Setup & Dependencies

Run the following script to load packages and dependencies

`run script.sh`

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import requests
import os

In [7]:
UNCERTAINTY_THRESHOLD = 0.5
DR_GRADES = {
    0: "No apparent DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR"
}

## Load Pre-trained Models

In [8]:
# load EyePACS straight from huggingface
from datasets import load_dataset

eyepacs = load_dataset("ctmedtech/EYEPACS", split="train")
print(eyepacs)

Dataset({
    features: ['image'],
    num_rows: 35111
})


In [ ]:
# transforms to apply images
def get_transforms():
    return transforms.Compose(
        [
            transforms.Resize((512, 512)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

# dataset
class DRDataset(Dataset):
    def __init__(self, data_source, img_dir=None, transform=None):
        self.data_source = data_source
        self.img_dir = img_dir
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data_source)
    
    def __getitem__(self, idx):
        row = self.data_source[idx]
        if isinstance(row, pd.Series):
            row = row.to_dict()

        if isinstance(row, dict) and 'image' in row:
            img = row['image']
        else:
            img_name = row['image_name']
            img_path = os.path.join(self.img_dir, img_name)
            img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = row['label']
        return img, label

# create the dataset and load
dataset = DRDataset(eyepacs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)